# DriftGuard — Security Risk Label Generation

This notebook applies transparent security-focused labeling functions to real
configuration changes.

It produces:

- Weak labels for model training
- Rule identifiers and explanations
- Label-confidence scores
- Conflict indicators
- A manually reviewable annotation queue
- Separate train, validation and test usage controls

Weak labels are used only as noisy training supervision. Final validation and
test performance must be measured on manually reviewed labels.

In [1]:
import os
import re
import sys
import json
import gzip
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm


print("=" * 72)
print("DRIFTGUARD — SECURITY LABEL GENERATION")
print("=" * 72)

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

CONFIGS_DIR = PROJECT_ROOT / "configs"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
LABELS_DIR = PROJECT_ROOT / "data" / "labels"
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"
LOGS_DIR = PROJECT_ROOT / "logs"

LABELS_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Python version   :", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))
print("Project root     :", PROJECT_ROOT)
print("Labels folder    :", LABELS_DIR)
print("Evaluation folder:", EVALUATION_DIR)

DRIFTGUARD — SECURITY LABEL GENERATION
Python version   : 3.11.15
Python executable: C:\Users\Lenovo\anaconda3\envs\driftguard\python.exe
Conda environment: driftguard
Project root     : C:\Users\Lenovo\Desktop\DriftGuard
Labels folder    : C:\Users\Lenovo\Desktop\DriftGuard\data\labels
Evaluation folder: C:\Users\Lenovo\Desktop\DriftGuard\data\evaluation


In [2]:
repository_config_path = CONFIGS_DIR / "repositories.json"

if not repository_config_path.exists():
    raise FileNotFoundError(
        f"Missing repository configuration:\n{repository_config_path}"
    )

with repository_config_path.open("r", encoding="utf-8") as file:
    repository_configuration = json.load(file)

REPOSITORIES = repository_configuration["repositories"]

repository_df = pd.DataFrame(REPOSITORIES)

display(
    repository_df[
        ["name", "split", "primary_config_types"]
    ]
)

print("Repositories loaded:", len(REPOSITORIES))

,name,split,primary_config_types
0,microservices_demo,train,"[kubernetes, helm, kustomize, terraform]"
1,kube_prometheus,train,"[kubernetes, jsonnet, yaml]"
2,terraform_aws_vpc,train,[terraform]
3,kubernetes_examples,validation,"[kubernetes, yaml]"
4,ansible_examples,test,"[ansible, nginx, yaml]"
5,terraform_eks_blueprints,test,"[terraform, kubernetes]"


Repositories loaded: 6


In [3]:
structural_output_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    structural_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diffs.jsonl.gz"
    )

    summary_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diff_summary.json"
    )

    expected_records = None

    if summary_path.exists():
        with summary_path.open("r", encoding="utf-8") as file:
            summary = json.load(file)

        expected_records = summary.get(
            "generated_diff_records"
        )

    structural_output_records.append(
        {
            "repository": repository_name,
            "split": repository["split"],
            "structural_file_exists": structural_path.exists(),
            "summary_exists": summary_path.exists(),
            "expected_records": expected_records,
            "file_size_mb": (
                round(
                    structural_path.stat().st_size / (1024 ** 2),
                    2,
                )
                if structural_path.exists()
                else None
            ),
        }
    )

structural_outputs_df = pd.DataFrame(
    structural_output_records
)

display(structural_outputs_df)

if not structural_outputs_df[
    "structural_file_exists"
].all():
    missing = structural_outputs_df.loc[
        ~structural_outputs_df["structural_file_exists"],
        "repository",
    ].tolist()

    raise FileNotFoundError(
        "Missing structural diff files for: "
        + ", ".join(missing)
    )

print("\nAll Notebook 03 outputs are available.")

,repository,split,structural_file_exists,summary_exists,expected_records,file_size_mb
0,microservices_demo,train,True,True,83933,10.64
1,kube_prometheus,train,True,True,130430,17.24
2,terraform_aws_vpc,train,True,True,14505,1.71
3,kubernetes_examples,validation,True,True,12854,1.62
4,ansible_examples,test,True,True,8940,1.09
5,terraform_eks_blueprints,test,True,True,109076,13.04



All Notebook 03 outputs are available.


In [4]:
LABEL_SETTINGS = {
    "force_regenerate": False,

    "valid_labels": [
        "benign",
        "low",
        "medium",
        "high",
        "critical",
        "unlabeled",
    ],

    # Only high-confidence training records will later be selected.
    "minimum_training_confidence": 0.85,

    # Manual annotation queue size by split.
    "manual_review_targets": {
        "train": 700,
        "validation": 400,
        "test": 700,
    },

    "random_seed": 42,
}

label_settings_path = (
    CONFIGS_DIR / "security_label_settings.json"
)

with label_settings_path.open("w", encoding="utf-8") as file:
    json.dump(
        LABEL_SETTINGS,
        file,
        indent=2,
    )

print(json.dumps(LABEL_SETTINGS, indent=2))
print("\nSaved to:", label_settings_path)

{
  "force_regenerate": false,
  "valid_labels": [
    "benign",
    "low",
    "medium",
    "high",
    "critical",
    "unlabeled"
  ],
  "minimum_training_confidence": 0.85,
  "manual_review_targets": {
    "train": 700,
    "validation": 400,
    "test": 700
  },
  "random_seed": 42
}

Saved to: C:\Users\Lenovo\Desktop\DriftGuard\configs\security_label_settings.json


In [6]:
SEVERITY_RANK = {
    "unlabeled": -1,
    "benign": 0,
    "low": 1,
    "medium": 2,
    "high": 3,
    "critical": 4,
}

RULE_DEFINITIONS = {
    "TLS_DISABLED": {
        "default_label": "critical",
        "description": "TLS or certificate verification was disabled.",
    },

    "AUTH_REMOVED": {
        "default_label": "critical",
        "description": "Authentication or authorization protection was removed.",
    },

    "DEFAULT_CREDENTIALS": {
        "default_label": "critical",
        "description": "A default, trivial or hard-coded credential was introduced.",
    },

    "OPEN_PORT_EXPOSURE": {
        "default_label": "high",
        "description": "A service or network range was exposed publicly.",
    },

    "PERMISSIVE_ACL": {
        "default_label": "high",
        "description": "Access control or runtime permissions became overly permissive.",
    },

    "INSECURE_PROTOCOL": {
        "default_label": "high",
        "description": "A secure protocol was replaced with an insecure protocol.",
    },

    "RESOURCE_LIMIT_REMOVED": {
        "default_label": "medium",
        "description": "A resource limit, quota or protection was removed.",
    },

    "UNCLASSIFIED_SENSITIVE_FIELD_CHANGE": {
        "default_label": "low",
        "description": "A sensitive configuration field changed without a stronger rule match.",
    },
}

RULE_PRIORITY = {
    "DEFAULT_CREDENTIALS": 8,
    "AUTH_REMOVED": 7,
    "TLS_DISABLED": 6,
    "OPEN_PORT_EXPOSURE": 5,
    "PERMISSIVE_ACL": 4,
    "INSECURE_PROTOCOL": 3,
    "RESOURCE_LIMIT_REMOVED": 2,
    "UNCLASSIFIED_SENSITIVE_FIELD_CHANGE": 1,
}

VALID_RULE_IDS = set(RULE_DEFINITIONS)

display(
    pd.DataFrame(
        [
            {
                "rule_id": rule_id,
                **definition,
            }
            for rule_id, definition
            in RULE_DEFINITIONS.items()
        ]
    )
)

,rule_id,default_label,description
0,TLS_DISABLED,critical,TLS or certificate verification was disabled.
1,AUTH_REMOVED,critical,Authentication or authorization protection was...
2,DEFAULT_CREDENTIALS,critical,"A default, trivial or hard-coded credential wa..."
3,OPEN_PORT_EXPOSURE,high,A service or network range was exposed publicly.
4,PERMISSIVE_ACL,high,Access control or runtime permissions became o...
5,INSECURE_PROTOCOL,high,A secure protocol was replaced with an insecur...
6,RESOURCE_LIMIT_REMOVED,medium,"A resource limit, quota or protection was remo..."
7,UNCLASSIFIED_SENSITIVE_FIELD_CHANGE,low,A sensitive configuration field changed withou...


In [7]:
TRUE_LIKE_VALUES = {
    "true",
    "1",
    "yes",
    "on",
    "enabled",
    "enable",
    "required",
}

FALSE_LIKE_VALUES = {
    "false",
    "0",
    "no",
    "off",
    "disabled",
    "disable",
    "none",
    "null",
    "nil",
    "~",
    "",
}

DEFAULT_CREDENTIAL_VALUES = {
    "admin",
    "admin123",
    "adminadmin",
    "password",
    "password123",
    "pass123",
    "changeme",
    "change_me",
    "default",
    "root",
    "rootroot",
    "guest",
    "test",
    "secret",
    "1234",
    "12345",
    "123456",
    "qwerty",
}

PUBLIC_NETWORK_VALUES = {
    "0.0.0.0",
    "0.0.0.0/0",
    "::",
    "::/0",
    "*",
    "all",
    "any",
    "anywhere",
    "world",
    "public",
}

TEMPLATE_MARKERS = [
    "${",
    "{{",
    "var.",
    "secretref",
    "secret_key_ref",
    "valuefrom",
    "vault:",
    "secretsmanager",
]


def normalize_text(value):
    """Normalize arbitrary values for rule matching."""

    if value is None:
        return ""

    text = str(value).strip().lower()

    if len(text) >= 2:
        if (
            text.startswith('"')
            and text.endswith('"')
        ) or (
            text.startswith("'")
            and text.endswith("'")
        ):
            text = text[1:-1].strip()

    text = re.sub(r"\s+", " ", text)

    return text


def contains_any(text, terms):
    """Return True when any term occurs in text."""

    normalized = normalize_text(text)

    return any(
        normalize_text(term) in normalized
        for term in terms
    )


def is_true_like(value):
    return normalize_text(value) in TRUE_LIKE_VALUES


def is_false_like(value):
    return normalize_text(value) in FALSE_LIKE_VALUES


def is_missing_value(value):
    return value is None or normalize_text(value) in {
        "",
        "none",
        "null",
        "nil",
        "~",
    }


def is_templated_secret(value):
    normalized = normalize_text(value)

    return any(
        marker in normalized
        for marker in TEMPLATE_MARKERS
    )


def build_record_context(record):
    """Create searchable text from relevant Diff Object fields."""

    fields = [
        record.get("field_path"),
        record.get("old_value"),
        record.get("new_value"),
        record.get("file_path"),
        record.get("configuration_type"),
        record.get("commit_message"),
    ]

    return " | ".join(
        normalize_text(value)
        for value in fields
        if value is not None
    )

In [8]:
SENSITIVE_FIELD_TERMS = {
    "auth",
    "authentication",
    "authorization",
    "authorisation",
    "rbac",
    "oauth",
    "oidc",
    "saml",
    "ldap",
    "token",
    "secret",
    "password",
    "passwd",
    "credential",
    "api_key",
    "apikey",
    "private_key",
    "tls",
    "ssl",
    "https",
    "certificate",
    "verify",
    "encrypt",
    "cipher",
    "port",
    "listen",
    "bind",
    "hostport",
    "nodeport",
    "cidr",
    "source_range",
    "security_group",
    "firewall",
    "ingress",
    "egress",
    "acl",
    "allow",
    "deny",
    "permission",
    "privileged",
    "privilege",
    "capability",
    "hostnetwork",
    "hostpid",
    "hostipc",
    "runasroot",
    "runasnonroot",
    "readonlyrootfilesystem",
    "limit",
    "quota",
    "resources",
    "cpu",
    "memory",
    "anonymous",
}


def find_sensitive_terms(record):
    """Return security-sensitive terms found in a record."""

    searchable_text = " ".join(
        [
            normalize_text(record.get("field_path")),
            normalize_text(record.get("file_path")),
            normalize_text(record.get("new_value")),
            normalize_text(record.get("old_value")),
        ]
    )

    matched_terms = sorted(
        term
        for term in SENSITIVE_FIELD_TERMS
        if term in searchable_text
    )

    return matched_terms


def is_sensitive_record(record):
    return len(find_sensitive_terms(record)) > 0

In [9]:
def create_rule_vote(
    rule_id,
    label,
    confidence,
    reason,
    evidence=None,
):
    """Create a standardized rule vote."""

    if rule_id not in VALID_RULE_IDS:
        raise ValueError(
            f"Unknown security rule: {rule_id}"
        )

    if label not in SEVERITY_RANK:
        raise ValueError(
            f"Unknown risk label: {label}"
        )

    return {
        "rule_id": rule_id,
        "label": label,
        "confidence": float(confidence),
        "reason": reason,
        "evidence": evidence,
    }

In [10]:
TLS_FIELD_TERMS = {
    "tls",
    "ssl",
    "https",
    "certificate",
    "cert_verify",
    "verify_ssl",
    "ssl_verify",
    "tls_verify",
    "insecure_skip_verify",
}

TLS_DISABLE_FIELD_TERMS = {
    "enabled",
    "enable",
    "required",
    "verify",
    "verification",
}


def evaluate_tls_disabled(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    old_value = record.get("old_value")
    new_value = record.get("new_value")
    operation = record.get("operation")

    tls_field = contains_any(
        field_path,
        TLS_FIELD_TERMS,
    )

    if not tls_field:
        return None

    skip_verify_field = (
        "insecure_skip_verify" in field_path
        or "insecureskipverify" in field_path
    )

    if skip_verify_field and is_true_like(new_value):
        return create_rule_vote(
            rule_id="TLS_DISABLED",
            label="critical",
            confidence=0.99,
            reason=(
                "TLS certificate verification was explicitly bypassed."
            ),
            evidence=f"{field_path}={new_value}",
        )

    explicit_disable = (
        is_false_like(new_value)
        and contains_any(
            field_path,
            TLS_DISABLE_FIELD_TERMS,
        )
    )

    deleted_protection = (
        operation == "deleted"
        and is_true_like(old_value)
    )

    if explicit_disable or deleted_protection:
        return create_rule_vote(
            rule_id="TLS_DISABLED",
            label="critical",
            confidence=0.98,
            reason=(
                "TLS or certificate verification protection "
                "was disabled or removed."
            ),
            evidence=(
                f"{field_path}: {old_value} -> {new_value}"
            ),
        )

    return None

In [11]:
AUTH_FIELD_TERMS = {
    "auth",
    "authentication",
    "authorization",
    "authorisation",
    "rbac",
    "oauth",
    "oidc",
    "saml",
    "ldap",
    "login_required",
    "token_required",
    "require_auth",
}


def evaluate_auth_removed(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    old_value = record.get("old_value")
    new_value = record.get("new_value")
    operation = record.get("operation")

    if not contains_any(
        field_path,
        AUTH_FIELD_TERMS,
    ):
        return None

    # Anonymous access becoming enabled is handled as permissive access.
    if "anonymous" in field_path:
        return None

    disabled_auth = is_false_like(new_value)

    deleted_auth = (
        operation == "deleted"
        and not is_missing_value(old_value)
    )

    if disabled_auth or deleted_auth:
        return create_rule_vote(
            rule_id="AUTH_REMOVED",
            label="critical",
            confidence=0.98,
            reason=(
                "Authentication or authorization protection "
                "was disabled or deleted."
            ),
            evidence=(
                f"{field_path}: {old_value} -> {new_value}"
            ),
        )

    return None

In [12]:
CREDENTIAL_FIELD_TERMS = {
    "password",
    "passwd",
    "credential",
    "secret",
    "token",
    "api_key",
    "apikey",
    "username",
    "user",
    "private_key",
}


def evaluate_default_credentials(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    new_value = normalize_text(
        record.get("new_value")
    )

    if not contains_any(
        field_path,
        CREDENTIAL_FIELD_TERMS,
    ):
        return None

    if not new_value:
        return None

    if is_templated_secret(new_value):
        return None

    compact_value = re.sub(
        r"[^a-z0-9]",
        "",
        new_value,
    )

    exact_default = (
        new_value in DEFAULT_CREDENTIAL_VALUES
        or compact_value in DEFAULT_CREDENTIAL_VALUES
    )

    credential_pair = bool(
        re.search(
            r"\b(admin|root|guest|test)"
            r"\s*[:=/]\s*"
            r"(admin|root|password|changeme|test|123456)\b",
            new_value,
        )
    )

    hardcoded_password_expression = bool(
        re.search(
            r"(password|passwd|secret|token)"
            r"\s*[:=]\s*"
            r"[\"']?"
            r"(admin|root|password|changeme|123456)"
            r"[\"']?",
            new_value,
        )
    )

    if (
        exact_default
        or credential_pair
        or hardcoded_password_expression
    ):
        return create_rule_vote(
            rule_id="DEFAULT_CREDENTIALS",
            label="critical",
            confidence=0.99,
            reason=(
                "A default, trivial or hard-coded credential "
                "was introduced."
            ),
            evidence=f"{field_path}={new_value}",
        )

    return None

In [13]:
NETWORK_EXPOSURE_TERMS = {
    "cidr",
    "cidr_blocks",
    "source_range",
    "source_ranges",
    "source_ip",
    "allowed_ip",
    "allowed_ips",
    "bind",
    "listen_address",
    "host",
    "interface",
    "ingress",
    "security_group",
    "firewall",
}


def evaluate_open_port_exposure(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    new_value = normalize_text(
        record.get("new_value")
    )

    if not contains_any(
        field_path,
        NETWORK_EXPOSURE_TERMS,
    ):
        return None

    public_value_found = any(
        public_value == new_value
        or public_value in new_value
        for public_value in PUBLIC_NETWORK_VALUES
    )

    public_phrase_found = bool(
        re.search(
            r"\b(allow all|public access|open to world|anywhere)\b",
            new_value,
        )
    )

    if public_value_found or public_phrase_found:
        label = (
            "critical"
            if (
                "0.0.0.0/0" in new_value
                or "::/0" in new_value
            )
            else "high"
        )

        confidence = (
            0.99 if label == "critical" else 0.96
        )

        return create_rule_vote(
            rule_id="OPEN_PORT_EXPOSURE",
            label=label,
            confidence=confidence,
            reason=(
                "Network access was expanded to a public "
                "or unrestricted address range."
            ),
            evidence=f"{field_path}={new_value}",
        )

    return None

In [14]:
PERMISSIVE_FIELD_TERMS = {
    "allow",
    "allowed",
    "acl",
    "permission",
    "privileged",
    "allowprivilegeescalation",
    "allow_privilege_escalation",
    "runasnonroot",
    "run_as_non_root",
    "hostnetwork",
    "host_network",
    "hostpid",
    "host_pid",
    "hostipc",
    "host_ipc",
    "readonlyrootfilesystem",
    "read_only_root_filesystem",
    "anonymous",
    "capabilities",
}

PERMISSIVE_VALUES = {
    "*",
    "all",
    "any",
    "everyone",
    "world",
    "anonymous",
    "allow_all",
    "allowall",
    "unrestricted",
}


def evaluate_permissive_acl(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    new_value = normalize_text(
        record.get("new_value")
    )

    if not contains_any(
        field_path,
        PERMISSIVE_FIELD_TERMS,
    ):
        return None

    privileged_enabled = (
        "privileged" in field_path
        and is_true_like(new_value)
    )

    privilege_escalation_enabled = (
        (
            "allowprivilegeescalation" in field_path
            or "allow_privilege_escalation" in field_path
        )
        and is_true_like(new_value)
    )

    host_namespace_enabled = (
        contains_any(
            field_path,
            {
                "hostnetwork",
                "host_network",
                "hostpid",
                "host_pid",
                "hostipc",
                "host_ipc",
            },
        )
        and is_true_like(new_value)
    )

    anonymous_enabled = (
        "anonymous" in field_path
        and is_true_like(new_value)
    )

    run_as_non_root_disabled = (
        contains_any(
            field_path,
            {
                "runasnonroot",
                "run_as_non_root",
            },
        )
        and is_false_like(new_value)
    )

    read_only_root_disabled = (
        contains_any(
            field_path,
            {
                "readonlyrootfilesystem",
                "read_only_root_filesystem",
            },
        )
        and is_false_like(new_value)
    )

    unrestricted_value = (
        new_value in PERMISSIVE_VALUES
        or bool(
            re.search(
                r"\b(allow all|everyone|all users|any user)\b",
                new_value,
            )
        )
    )

    dangerous = any(
        [
            privileged_enabled,
            privilege_escalation_enabled,
            host_namespace_enabled,
            anonymous_enabled,
            run_as_non_root_disabled,
            read_only_root_disabled,
            unrestricted_value,
        ]
    )

    if dangerous:
        critical = (
            privileged_enabled
            or privilege_escalation_enabled
            or anonymous_enabled
        )

        return create_rule_vote(
            rule_id="PERMISSIVE_ACL",
            label=(
                "critical" if critical else "high"
            ),
            confidence=(
                0.98 if critical else 0.94
            ),
            reason=(
                "Access control or runtime security "
                "became overly permissive."
            ),
            evidence=f"{field_path}={new_value}",
        )

    return None

In [15]:
INSECURE_PROTOCOL_PATTERNS = [
    r"\bhttp://",
    r"\bftp://",
    r"\btelnet://",
    r"\brlogin\b",
    r"\bplaintext\b",
    r"\bcleartext\b",
    r"\bno[_-]?tls\b",
]


def evaluate_insecure_protocol(record):
    old_value = normalize_text(
        record.get("old_value")
    )

    new_value = normalize_text(
        record.get("new_value")
    )

    secure_to_insecure = (
        "https://" in old_value
        and "http://" in new_value
        and "https://" not in new_value
    )

    insecure_pattern_found = any(
        re.search(pattern, new_value)
        for pattern in INSECURE_PROTOCOL_PATTERNS
    )

    if secure_to_insecure or insecure_pattern_found:
        return create_rule_vote(
            rule_id="INSECURE_PROTOCOL",
            label="high",
            confidence=0.95,
            reason=(
                "A secure communication protocol was "
                "replaced by an insecure alternative."
            ),
            evidence=(
                f"{old_value} -> {new_value}"
            ),
        )

    return None

In [16]:
RESOURCE_LIMIT_TERMS = {
    "resources.limits",
    ".limits.",
    ".limit.",
    "cpu_limit",
    "memory_limit",
    "quota",
    "max_connections",
    "rate_limit",
    "ratelimit",
    "request_limit",
}

UNLIMITED_VALUES = {
    "0",
    "-1",
    "none",
    "null",
    "unlimited",
    "infinite",
    "infinity",
    "disabled",
}


def evaluate_resource_limit_removed(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    operation = record.get("operation")
    old_value = record.get("old_value")
    new_value = normalize_text(
        record.get("new_value")
    )

    if not contains_any(
        field_path,
        RESOURCE_LIMIT_TERMS,
    ):
        return None

    deleted_limit = (
        operation == "deleted"
        and not is_missing_value(old_value)
    )

    unlimited_limit = (
        new_value in UNLIMITED_VALUES
    )

    if deleted_limit or unlimited_limit:
        return create_rule_vote(
            rule_id="RESOURCE_LIMIT_REMOVED",
            label="medium",
            confidence=0.91,
            reason=(
                "A resource, quota or rate limit was "
                "removed or made unlimited."
            ),
            evidence=(
                f"{field_path}: {old_value} -> {new_value}"
            ),
        )

    return None

In [17]:
BENIGN_FIELD_TERMS = {
    ".metadata.labels.",
    ".metadata.annotations.",
    ".image",
    ".imagepullpolicy",
    ".tag",
    ".version",
    ".replicas",
    ".description",
    ".displayname",
    ".display_name",
    ".chart",
    ".name",
    "instructions.copy",
    "instructions.workdir",
    "instructions.label",
}

DANGEROUS_VALUE_PATTERNS = [
    r"0\.0\.0\.0/0",
    r"::/0",
    r"\bprivileged\b",
    r"\ballow all\b",
    r"\badmin:admin\b",
    r"\bpassword\s*[:=]",
    r"\bhttp://",
    r"\bftp://",
]


def evaluate_benign_candidate(record):
    field_path = normalize_text(
        record.get("field_path")
    )

    context = build_record_context(record)

    if is_sensitive_record(record):
        return None

    known_benign_field = contains_any(
        field_path,
        BENIGN_FIELD_TERMS,
    )

    dangerous_context = any(
        re.search(pattern, context)
        for pattern in DANGEROUS_VALUE_PATTERNS
    )

    if known_benign_field and not dangerous_context:
        return {
            "label": "benign",
            "confidence": 0.92,
            "reason": (
                "The change modifies a common non-security "
                "metadata, naming, version or scaling field."
            ),
        }

    # Other non-sensitive structural changes receive lower confidence.
    if (
        record.get("parser_mode") == "structured"
        and not dangerous_context
    ):
        return {
            "label": "benign",
            "confidence": 0.72,
            "reason": (
                "No security-sensitive field or dangerous "
                "value pattern was detected."
            ),
        }

    return None

In [18]:
SECURITY_RULE_FUNCTIONS = [
    evaluate_default_credentials,
    evaluate_auth_removed,
    evaluate_tls_disabled,
    evaluate_open_port_exposure,
    evaluate_permissive_acl,
    evaluate_insecure_protocol,
    evaluate_resource_limit_removed,
]


def apply_security_labels(record):
    """Apply security rules and resolve multiple votes."""

    votes = []

    for rule_function in SECURITY_RULE_FUNCTIONS:
        vote = rule_function(record)

        if vote is not None:
            votes.append(vote)

    sensitive_terms = find_sensitive_terms(record)

    if not votes and sensitive_terms:
        votes.append(
            create_rule_vote(
                rule_id=(
                    "UNCLASSIFIED_SENSITIVE_FIELD_CHANGE"
                ),
                label="low",
                confidence=0.76,
                reason=(
                    "A security-sensitive field changed, "
                    "but no stronger risk pattern matched."
                ),
                evidence=", ".join(
                    sensitive_terms[:10]
                ),
            )
        )

    if votes:
        sorted_votes = sorted(
            votes,
            key=lambda vote: (
                SEVERITY_RANK[vote["label"]],
                vote["confidence"],
                RULE_PRIORITY[vote["rule_id"]],
            ),
            reverse=True,
        )

        primary_vote = sorted_votes[0]

        matched_rule_ids = [
            vote["rule_id"]
            for vote in sorted_votes
        ]

        matched_labels = {
            vote["label"]
            for vote in sorted_votes
        }

        rule_conflict = (
            len(matched_rule_ids) > 1
            or len(matched_labels) > 1
        )

        weak_label = primary_vote["label"]
        confidence = primary_vote["confidence"]
        primary_rule_id = primary_vote["rule_id"]
        reason = primary_vote["reason"]
        label_source = "security_rule"

    else:
        benign_result = evaluate_benign_candidate(
            record
        )

        if benign_result is not None:
            weak_label = benign_result["label"]
            confidence = benign_result["confidence"]
            primary_rule_id = None
            matched_rule_ids = []
            rule_conflict = False
            reason = benign_result["reason"]
            label_source = "benign_heuristic"

        else:
            weak_label = "unlabeled"
            confidence = 0.0
            primary_rule_id = None
            matched_rule_ids = []
            rule_conflict = False
            reason = (
                "Insufficient evidence for a reliable "
                "security or benign weak label."
            )
            label_source = "unlabeled"

    dataset_split = record.get(
        "dataset_split"
    )

    if dataset_split == "train":
        label_usage = "weak_training_candidate"
    elif dataset_split == "validation":
        label_usage = "validation_review_only"
    else:
        label_usage = "test_review_only"

    needs_manual_review = bool(
        weak_label == "unlabeled"
        or confidence < 0.90
        or rule_conflict
        or record.get("parser_mode") == "line_fallback"
    )

    return {
        "weak_label": weak_label,
        "weak_label_confidence": float(confidence),
        "severity_rank": SEVERITY_RANK[weak_label],
        "primary_rule_id": primary_rule_id,
        "matched_rule_ids": matched_rule_ids,
        "matched_rule_count": len(matched_rule_ids),
        "rule_conflict": rule_conflict,
        "weak_label_reason": reason,
        "label_source": label_source,
        "label_usage": label_usage,
        "sensitive_terms": sensitive_terms,
        "sensitive_term_count": len(sensitive_terms),
        "needs_manual_review": needs_manual_review,
    }

In [19]:
rule_test_cases = [
    {
        "name": "TLS disabled",
        "expected_rule": "TLS_DISABLED",
        "record": {
            "field_path": "$.spec.tls.enabled",
            "old_value": "true",
            "new_value": "false",
            "operation": "modified",
            "file_path": "deployment.yaml",
            "configuration_type": "kubernetes",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Authentication removed",
        "expected_rule": "AUTH_REMOVED",
        "record": {
            "field_path": "$.authentication.enabled",
            "old_value": "true",
            "new_value": None,
            "operation": "deleted",
            "file_path": "config.yaml",
            "configuration_type": "yaml",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Default password",
        "expected_rule": "DEFAULT_CREDENTIALS",
        "record": {
            "field_path": "$.database.password",
            "old_value": None,
            "new_value": "admin123",
            "operation": "added",
            "file_path": "values.yaml",
            "configuration_type": "helm",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Public CIDR",
        "expected_rule": "OPEN_PORT_EXPOSURE",
        "record": {
            "field_path": "$.ingress.cidr_blocks[0]",
            "old_value": "10.0.0.0/8",
            "new_value": "0.0.0.0/0",
            "operation": "modified",
            "file_path": "security.tf",
            "configuration_type": "terraform",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Privileged container",
        "expected_rule": "PERMISSIVE_ACL",
        "record": {
            "field_path": "$.securityContext.privileged",
            "old_value": "false",
            "new_value": "true",
            "operation": "modified",
            "file_path": "deployment.yaml",
            "configuration_type": "kubernetes",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "HTTPS changed to HTTP",
        "expected_rule": "INSECURE_PROTOCOL",
        "record": {
            "field_path": "$.backend.endpoint",
            "old_value": "https://api.example.com",
            "new_value": "http://api.example.com",
            "operation": "modified",
            "file_path": "config.yaml",
            "configuration_type": "yaml",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Memory limit removed",
        "expected_rule": "RESOURCE_LIMIT_REMOVED",
        "record": {
            "field_path": "$.resources.limits.memory",
            "old_value": "512Mi",
            "new_value": None,
            "operation": "deleted",
            "file_path": "deployment.yaml",
            "configuration_type": "kubernetes",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Unclassified sensitive field",
        "expected_rule": (
            "UNCLASSIFIED_SENSITIVE_FIELD_CHANGE"
        ),
        "record": {
            "field_path": "$.token.expiry_seconds",
            "old_value": "3600",
            "new_value": "7200",
            "operation": "modified",
            "file_path": "config.yaml",
            "configuration_type": "yaml",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },

    {
        "name": "Benign replicas change",
        "expected_rule": None,
        "record": {
            "field_path": "$.spec.replicas",
            "old_value": "2",
            "new_value": "4",
            "operation": "modified",
            "file_path": "deployment.yaml",
            "configuration_type": "kubernetes",
            "parser_mode": "structured",
            "commit_message": "",
            "dataset_split": "train",
        },
    },
]

rule_test_results = []

for test_case in rule_test_cases:
    result = apply_security_labels(
        test_case["record"]
    )

    rule_test_results.append(
        {
            "test": test_case["name"],
            "expected_rule": test_case[
                "expected_rule"
            ],
            "actual_rule": result[
                "primary_rule_id"
            ],
            "weak_label": result["weak_label"],
            "confidence": result[
                "weak_label_confidence"
            ],
            "passed": (
                result["primary_rule_id"]
                == test_case["expected_rule"]
            ),
        }
    )

rule_test_df = pd.DataFrame(
    rule_test_results
)

display(rule_test_df)

assert rule_test_df["passed"].all()

print("All security labeling tests: PASSED")

,test,expected_rule,actual_rule,weak_label,confidence,passed
0,TLS disabled,TLS_DISABLED,TLS_DISABLED,critical,0.98,True
1,Authentication removed,AUTH_REMOVED,AUTH_REMOVED,critical,0.98,True
2,Default password,DEFAULT_CREDENTIALS,DEFAULT_CREDENTIALS,critical,0.99,True
3,Public CIDR,OPEN_PORT_EXPOSURE,OPEN_PORT_EXPOSURE,critical,0.99,True
4,Privileged container,PERMISSIVE_ACL,PERMISSIVE_ACL,critical,0.98,True
5,HTTPS changed to HTTP,INSECURE_PROTOCOL,INSECURE_PROTOCOL,high,0.95,True
6,Memory limit removed,RESOURCE_LIMIT_REMOVED,RESOURCE_LIMIT_REMOVED,medium,0.91,True
7,Unclassified sensitive field,UNCLASSIFIED_SENSITIVE_FIELD_CHANGE,UNCLASSIFIED_SENSITIVE_FIELD_CHANGE,low,0.76,True
8,Benign replicas change,NaN,NaN,benign,0.92,True


All security labeling tests: PASSED


In [21]:
def generate_labels_for_repository(
    repository_config,
    force_regenerate=False,
):
    """Generate weak security labels for one repository."""

    repository_name = repository_config["name"]
    dataset_split = repository_config["split"]

    input_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diffs.jsonl.gz"
    )

    structural_summary_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diff_summary.json"
    )

    output_path = (
        LABELS_DIR
        / f"{repository_name}_weak_labels.jsonl.gz"
    )

    output_summary_path = (
        LABELS_DIR
        / f"{repository_name}_weak_label_summary.json"
    )

    temporary_path = Path(
        str(output_path) + ".tmp"
    )

    if (
        output_path.exists()
        and output_summary_path.exists()
        and not force_regenerate
    ):
        print(
            f"{repository_name}: existing label output "
            "found; skipping."
        )

        with output_summary_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Missing structural diff file: {input_path}"
        )

    expected_records = None

    if structural_summary_path.exists():
        with structural_summary_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            structural_summary = json.load(file)

        expected_records = structural_summary.get(
            "generated_diff_records"
        )

    if temporary_path.exists():
        temporary_path.unlink()

    records_processed = 0
    records_written = 0
    processing_errors = 0

    label_counts = Counter()
    rule_counts = Counter()
    source_counts = Counter()
    parser_mode_counts = Counter()
    confidence_bands = Counter()
    manual_review_counts = Counter()

    print("\n" + "=" * 72)
    print(f"Repository : {repository_name}")
    print(f"Split      : {dataset_split}")
    print(
        "Expected   :",
        (
            f"{expected_records:,}"
            if expected_records is not None
            else "Unknown"
        ),
    )
    print("=" * 72)

    with gzip.open(
        input_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file, gzip.open(
        temporary_path,
        mode="wt",
        encoding="utf-8",
    ) as output_file:

        progress_bar = tqdm(
            input_file,
            total=expected_records,
            desc=repository_name,
            unit="diff",
            dynamic_ncols=True,
        )

        for line in progress_bar:
            records_processed += 1

            try:
                record = json.loads(line)

                label_result = apply_security_labels(
                    record
                )

                output_record = {
                    **record,
                    **label_result,
                    "weak_label_generated_at_utc":
                        datetime.now(
                            timezone.utc
                        ).isoformat(),
                }

                output_file.write(
                    json.dumps(
                        output_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

                records_written += 1

                weak_label = label_result[
                    "weak_label"
                ]

                confidence = label_result[
                    "weak_label_confidence"
                ]

                label_counts[weak_label] += 1
                source_counts[
                    label_result["label_source"]
                ] += 1

                parser_mode_counts[
                    record.get(
                        "parser_mode",
                        "unknown",
                    )
                ] += 1

                if label_result[
                    "primary_rule_id"
                ]:
                    rule_counts[
                        label_result[
                            "primary_rule_id"
                        ]
                    ] += 1

                if confidence >= 0.95:
                    confidence_bands["0.95-1.00"] += 1
                elif confidence >= 0.85:
                    confidence_bands["0.85-0.95"] += 1
                elif confidence > 0:
                    confidence_bands["0.01-0.85"] += 1
                else:
                    confidence_bands["0.00"] += 1

                manual_review_counts[
                    str(
                        label_result[
                            "needs_manual_review"
                        ]
                    )
                ] += 1

            except Exception:
                processing_errors += 1

            if (
                records_processed % 500 == 0
                or records_processed == 1
            ):
                progress_bar.set_postfix(
                    labeled=records_written,
                    risky=(
                        label_counts["low"]
                        + label_counts["medium"]
                        + label_counts["high"]
                        + label_counts["critical"]
                    ),
                    benign=label_counts["benign"],
                    unlabeled=label_counts["unlabeled"],
                    errors=processing_errors,
                )

        progress_bar.close()

    os.replace(
        temporary_path,
        output_path,
    )

    summary = {
        "repository": repository_name,
        "dataset_split": dataset_split,
        "input_path": str(input_path),
        "output_path": str(output_path),
        "expected_records": expected_records,
        "records_processed": records_processed,
        "records_written": records_written,
        "processing_errors": processing_errors,
        "label_counts": dict(label_counts),
        "rule_counts": dict(rule_counts),
        "label_source_counts": dict(source_counts),
        "parser_mode_counts": dict(
            parser_mode_counts
        ),
        "confidence_bands": dict(
            confidence_bands
        ),
        "manual_review_counts": dict(
            manual_review_counts
        ),
        "generated_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    with output_summary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    print("\nLabeling completed")
    print("Records processed:", f"{records_processed:,}")
    print("Records written  :", f"{records_written:,}")
    print("Errors           :", f"{processing_errors:,}")
    print("Label counts     :", dict(label_counts))

    return summary

In [22]:
weak_label_summaries = []

for repository_config in REPOSITORIES:
    summary = generate_labels_for_repository(
        repository_config=repository_config,
        force_regenerate=LABEL_SETTINGS[
            "force_regenerate"
        ],
    )

    weak_label_summaries.append(summary)

print("\nAll repository labeling stages completed.")


Repository : microservices_demo
Split      : train
Expected   : 83,933


microservices_demo: 100%|██████████| 83933/83933 [01:02<00:00, 1349.48diff/s, benign=38537, errors=0, labeled=83500, risky=35614, unlabeled=9349]



Labeling completed
Records processed: 83,933
Records written  : 83,933
Errors           : 0
Label counts     : {'benign': 38921, 'low': 34880, 'high': 378, 'medium': 174, 'unlabeled': 9349, 'critical': 231}

Repository : kube_prometheus
Split      : train
Expected   : 130,430


kube_prometheus: 100%|██████████| 130430/130430 [01:45<00:00, 1241.23diff/s, benign=78816, errors=0, labeled=130000, risky=39590, unlabeled=11594]



Labeling completed
Records processed: 130,430
Records written  : 130,430
Errors           : 0
Label counts     : {'benign': 78985, 'low': 38826, 'critical': 580, 'unlabeled': 11594, 'high': 346, 'medium': 99}

Repository : terraform_aws_vpc
Split      : train
Expected   : 14,505


...


Labeling completed
Records processed: 12,854
Records written  : 12,854
Errors           : 0
Label counts     : {'benign': 8640, 'low': 3194, 'high': 68, 'critical': 40, 'unlabeled': 871, 'medium': 41}

Repository : ansible_examples
Split      : test
Expected   : 8,940


ansible_examples: 100%|██████████| 8940/8940 [00:06<00:00, 1450.61diff/s, benign=6548, errors=0, labeled=8500, risky=1271, unlabeled=681]



Labeling completed
Records processed: 8,940
Records written  : 8,940
Errors           : 0
Label counts     : {'benign': 6951, 'low': 1058, 'unlabeled': 682, 'high': 110, 'critical': 139}

Repository : terraform_eks_blueprints
Split      : test
Expected   : 109,076


terraform_eks_blueprints: 100%|██████████| 109076/109076 [01:25<00:00, 1269.98diff/s, benign=77026, errors=0, labeled=109000, risky=25029, unlabeled=6945]


Labeling completed
Records processed: 109,076
Records written  : 109,076
Errors           : 0
Label counts     : {'benign': 77092, 'low': 24157, 'high': 226, 'medium': 116, 'critical': 540, 'unlabeled': 6945}

All repository labeling stages completed.


In [23]:
summary_records = []

for summary in weak_label_summaries:
    label_counts = summary["label_counts"]
    rule_counts = summary["rule_counts"]

    risky_count = sum(
        label_counts.get(label, 0)
        for label in [
            "low",
            "medium",
            "high",
            "critical",
        ]
    )

    summary_records.append(
        {
            "repository": summary["repository"],
            "split": summary["dataset_split"],
            "records": summary["records_written"],
            "benign": label_counts.get(
                "benign",
                0,
            ),
            "low": label_counts.get(
                "low",
                0,
            ),
            "medium": label_counts.get(
                "medium",
                0,
            ),
            "high": label_counts.get(
                "high",
                0,
            ),
            "critical": label_counts.get(
                "critical",
                0,
            ),
            "unlabeled": label_counts.get(
                "unlabeled",
                0,
            ),
            "risky_total": risky_count,
            "processing_errors":
                summary["processing_errors"],
        }
    )

weak_label_summary_df = pd.DataFrame(
    summary_records
)

display(weak_label_summary_df)

print(
    "\nTotal records:",
    f"{weak_label_summary_df['records'].sum():,}",
)

,repository,split,records,benign,low,medium,high,critical,unlabeled,risky_total,processing_errors
0,microservices_demo,train,83933,38921,34880,174,378,231,9349,35663,0
1,kube_prometheus,train,130430,78985,38826,99,346,580,11594,39851,0
2,terraform_aws_vpc,train,14505,10566,3667,0,252,16,4,3935,0
3,kubernetes_examples,validation,12854,8640,3194,41,68,40,871,3343,0
4,ansible_examples,test,8940,6951,1058,0,110,139,682,1307,0
5,terraform_eks_blueprints,test,109076,77092,24157,116,226,540,6945,25039,0



Total records: 359,738


In [24]:
COMPACT_COLUMNS = [
    "diff_id",
    "source_record_id",
    "repository",
    "dataset_split",
    "commit_hash",
    "commit_author_date",
    "commit_message",
    "file_path",
    "configuration_type",
    "parser_name",
    "parser_mode",
    "operation",
    "field_path",
    "old_value",
    "new_value",
    "change_signature_sha256",
    "weak_label",
    "weak_label_confidence",
    "severity_rank",
    "primary_rule_id",
    "matched_rule_ids",
    "matched_rule_count",
    "rule_conflict",
    "weak_label_reason",
    "label_source",
    "label_usage",
    "sensitive_terms",
    "sensitive_term_count",
    "needs_manual_review",
]


def preview_value(value, maximum_characters=700):
    if value is None:
        return None

    text = str(value)

    if len(text) <= maximum_characters:
        return text

    return (
        text[:maximum_characters]
        + " ... [PREVIEW TRUNCATED]"
    )


compact_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    labeled_path = (
        LABELS_DIR
        / f"{repository_name}_weak_labels.jsonl.gz"
    )

    with gzip.open(
        labeled_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file:

        for line in tqdm(
            input_file,
            desc=f"Loading {repository_name}",
            unit="record",
        ):
            record = json.loads(line)

            compact_record = {
                column: record.get(column)
                for column in COMPACT_COLUMNS
            }

            compact_record["old_value"] = preview_value(
                compact_record["old_value"]
            )

            compact_record["new_value"] = preview_value(
                compact_record["new_value"]
            )

            compact_record[
                "matched_rule_ids"
            ] = json.dumps(
                compact_record[
                    "matched_rule_ids"
                ],
                ensure_ascii=False,
            )

            compact_record[
                "sensitive_terms"
            ] = json.dumps(
                compact_record[
                    "sensitive_terms"
                ],
                ensure_ascii=False,
            )

            compact_records.append(
                compact_record
            )

weak_label_manifest = pd.DataFrame(
    compact_records
)

print(
    "Weak label manifest shape:",
    weak_label_manifest.shape,
)

display(weak_label_manifest.head())

Loading microservices_demo: 83933record [00:06, 13199.58record/s]
Loading kube_prometheus: 130430record [00:09, 13150.99record/s]
Loading terraform_aws_vpc: 14505record [00:01, 13325.56record/s]
Loading kubernetes_examples: 12854record [00:00, 14401.29record/s]
Loading ansible_examples: 8940record [00:00, 14328.86record/s]
Loading terraform_eks_blueprints: 109076record [00:09, 11838.37record/s]


Weak label manifest shape: (359738, 29)


,diff_id,source_record_id,repository,dataset_split,commit_hash,commit_author_date,commit_message,file_path,configuration_type,parser_name,...,primary_rule_id,matched_rule_ids,matched_rule_count,rule_conflict,weak_label_reason,label_source,label_usage,sensitive_terms,sensitive_term_count,needs_manual_review
0,c820a8520734fc4ecbbbcd59610480fca6eb145bfbf626...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/Chart.yaml,helm,yaml,...,NaN,[],0,False,No security-sensitive field or dangerous value...,benign_heuristic,weak_training_candidate,[],0,True
1,2778bac3137cbec71eb3ced76ea3d38addea260fd6a570...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/Chart.yaml,helm,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
2,dd772049c2639bec5386ebd1d70965302edda15fd2585f...,4ada560261335ebf7d43ecb68fcd56a2f0f3d380286a6b...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/values.yaml,helm,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
3,37a016215756d4ed90964a7487ef10b3ed80d5a4d3e8f2...,3cb9882df2d988155d0ea9c634473aa2bb22ba373bad1b...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,kustomize/base/adservice.yaml,kustomize,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
4,de79a27b115b18a990fedc06348d12fe1e6318c8fe5f7c...,c213b0bf8781befc5354bd284fa14c4cee54b360bb75a5...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,kustomize/base/cartservice.yaml,kustomize,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False


In [25]:
manifest_path = (
    LABELS_DIR
    / "weak_label_manifest.csv.gz"
)

summary_csv_path = (
    LABELS_DIR
    / "weak_label_repository_summary.csv"
)

summary_json_path = (
    LABELS_DIR
    / "weak_label_repository_summary.json"
)

weak_label_manifest.to_csv(
    manifest_path,
    index=False,
    compression="gzip",
)

weak_label_summary_df.to_csv(
    summary_csv_path,
    index=False,
)

with summary_json_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        weak_label_summaries,
        file,
        indent=2,
    )

print("Saved weak label manifest:")
print(manifest_path)

print("\nSaved summaries:")
print(summary_csv_path)
print(summary_json_path)

Saved weak label manifest:
C:\Users\Lenovo\Desktop\DriftGuard\data\labels\weak_label_manifest.csv.gz

Saved summaries:
C:\Users\Lenovo\Desktop\DriftGuard\data\labels\weak_label_repository_summary.csv
C:\Users\Lenovo\Desktop\DriftGuard\data\labels\weak_label_repository_summary.json


In [26]:
label_distribution = (
    weak_label_manifest
    .groupby(
        [
            "dataset_split",
            "weak_label",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "records"})
)

label_distribution[
    "percentage_within_split"
] = (
    label_distribution["records"]
    / label_distribution.groupby(
        "dataset_split"
    )["records"].transform("sum")
    * 100
)

display(
    label_distribution.sort_values(
        [
            "dataset_split",
            "records",
        ],
        ascending=[True, False],
    )
)

,dataset_split,weak_label,records,percentage_within_split
0,test,benign,84043,71.213225
3,test,low,25215,21.365747
5,test,unlabeled,7627,6.462683
1,test,critical,679,0.575346
2,test,high,336,0.284707
4,test,medium,116,0.098292
6,train,benign,128472,56.133667
9,train,low,77373,33.806823
11,train,unlabeled,20947,9.152437
8,train,high,976,0.426447


In [27]:
rule_distribution = (
    weak_label_manifest[
        weak_label_manifest[
            "primary_rule_id"
        ].notna()
    ]
    .groupby(
        [
            "dataset_split",
            "primary_rule_id",
            "weak_label",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "records"})
    .sort_values(
        [
            "dataset_split",
            "records",
        ],
        ascending=[True, False],
    )
)

display(rule_distribution)

,dataset_split,primary_rule_id,weak_label,records
9,test,UNCLASSIFIED_SENSITIVE_FIELD_CHANGE,low,25215
0,test,AUTH_REMOVED,critical,457
4,test,OPEN_PORT_EXPOSURE,high,258
1,test,DEFAULT_CREDENTIALS,critical,135
7,test,RESOURCE_LIMIT_REMOVED,medium,116
2,test,INSECURE_PROTOCOL,high,62
3,test,OPEN_PORT_EXPOSURE,critical,60
8,test,TLS_DISABLED,critical,24
6,test,PERMISSIVE_ACL,high,16
5,test,PERMISSIVE_ACL,critical,3


In [28]:
confidence_summary = (
    weak_label_manifest
    .groupby(
        [
            "dataset_split",
            "weak_label",
        ]
    )[
        "weak_label_confidence"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

display(confidence_summary)

print("\nRule conflicts:")

conflict_summary = (
    weak_label_manifest
    .groupby("dataset_split")
    .agg(
        total_records=("diff_id", "count"),
        conflict_records=(
            "rule_conflict",
            "sum",
        ),
        manual_review_records=(
            "needs_manual_review",
            "sum",
        ),
    )
)

conflict_summary[
    "conflict_rate"
] = (
    conflict_summary["conflict_records"]
    / conflict_summary["total_records"]
)

conflict_summary[
    "manual_review_rate"
] = (
    conflict_summary[
        "manual_review_records"
    ]
    / conflict_summary["total_records"]
)

display(conflict_summary)

,dataset_split,weak_label,count,mean,median,min,max
0,test,benign,84043,0.756103,0.72,0.72,0.92
1,test,critical,679,0.982916,0.98,0.98,0.99
2,test,high,336,0.957202,0.96,0.94,0.96
3,test,low,25215,0.760000,0.76,0.76,0.76
4,test,medium,116,0.910000,0.91,0.91,0.91
5,test,unlabeled,7627,0.000000,0.00,0.00,0.00
6,train,benign,128472,0.786407,0.72,0.72,0.92
7,train,critical,827,0.980605,0.98,0.98,0.99
8,train,high,976,0.950318,0.95,0.94,0.96
9,train,low,77373,0.760000,0.76,0.76,0.76



Rule conflicts:


,total_records,conflict_records,manual_review_records,conflict_rate,manual_review_rate
dataset_split,,,,,
test,118016,0,101714,0.000000,0.861866
train,228868,1,184140,0.000004,0.804569
validation,12854,0,8776,0.000000,0.682745


In [29]:
minimum_confidence = LABEL_SETTINGS[
    "minimum_training_confidence"
]

high_confidence_training_candidates = (
    weak_label_manifest[
        (
            weak_label_manifest[
                "dataset_split"
            ] == "train"
        )
        &
        (
            weak_label_manifest[
                "weak_label"
            ] != "unlabeled"
        )
        &
        (
            weak_label_manifest[
                "weak_label_confidence"
            ] >= minimum_confidence
        )
    ]
    .copy()
)

print(
    "High-confidence training candidates:",
    f"{len(high_confidence_training_candidates):,}",
)

training_candidate_distribution = (
    high_confidence_training_candidates[
        "weak_label"
    ]
    .value_counts()
    .rename_axis("weak_label")
    .reset_index(name="records")
)

display(training_candidate_distribution)

print(
    "\nValidation and test records are not included "
    "in this training candidate set."
)

High-confidence training candidates: 44,733


,weak_label,records
0,benign,42657
1,high,976
2,critical,827
3,medium,273



Validation and test records are not included in this training candidate set.


In [30]:
def balanced_annotation_sample(
    dataframe,
    target_size,
    random_seed,
):
    """
    Sample across repository, weak label and parser mode.

    Duplicate change signatures are removed before sampling.
    """

    if dataframe.empty:
        return dataframe.copy()

    candidates = (
        dataframe
        .drop_duplicates(
            subset=[
                "change_signature_sha256"
            ]
        )
        .copy()
    )

    if len(candidates) <= target_size:
        return candidates

    candidates["annotation_stratum"] = (
        candidates["repository"].astype(str)
        + "|"
        + candidates["weak_label"].astype(str)
        + "|"
        + candidates["parser_mode"].astype(str)
    )

    grouped = list(
        candidates.groupby(
            "annotation_stratum",
            sort=True,
        )
    )

    base_per_group = max(
        1,
        target_size // max(len(grouped), 1),
    )

    sampled_parts = []

    for group_index, (_, group_df) in enumerate(grouped):
        group_sample_size = min(
            len(group_df),
            base_per_group,
        )

        sampled_parts.append(
            group_df.sample(
                n=group_sample_size,
                random_state=(
                    random_seed + group_index
                ),
            )
        )

    sampled = pd.concat(
        sampled_parts,
        ignore_index=False,
    )

    sampled = sampled.drop_duplicates(
        subset=["diff_id"]
    )

    if len(sampled) < target_size:
        remaining = candidates[
            ~candidates["diff_id"].isin(
                sampled["diff_id"]
            )
        ]

        additional_size = min(
            target_size - len(sampled),
            len(remaining),
        )

        if additional_size > 0:
            additional = remaining.sample(
                n=additional_size,
                random_state=random_seed + 10_000,
            )

            sampled = pd.concat(
                [
                    sampled,
                    additional,
                ],
                ignore_index=False,
            )

    sampled = sampled.sample(
        frac=1,
        random_state=random_seed,
    )

    return sampled.head(
        target_size
    ).drop(
        columns=["annotation_stratum"],
        errors="ignore",
    )

In [31]:
def balanced_annotation_sample(
    dataframe,
    target_size,
    random_seed,
):
    """
    Sample across repository, weak label and parser mode.

    Duplicate change signatures are removed before sampling.
    """

    if dataframe.empty:
        return dataframe.copy()

    candidates = (
        dataframe
        .drop_duplicates(
            subset=[
                "change_signature_sha256"
            ]
        )
        .copy()
    )

    if len(candidates) <= target_size:
        return candidates

    candidates["annotation_stratum"] = (
        candidates["repository"].astype(str)
        + "|"
        + candidates["weak_label"].astype(str)
        + "|"
        + candidates["parser_mode"].astype(str)
    )

    grouped = list(
        candidates.groupby(
            "annotation_stratum",
            sort=True,
        )
    )

    base_per_group = max(
        1,
        target_size // max(len(grouped), 1),
    )

    sampled_parts = []

    for group_index, (_, group_df) in enumerate(grouped):
        group_sample_size = min(
            len(group_df),
            base_per_group,
        )

        sampled_parts.append(
            group_df.sample(
                n=group_sample_size,
                random_state=(
                    random_seed + group_index
                ),
            )
        )

    sampled = pd.concat(
        sampled_parts,
        ignore_index=False,
    )

    sampled = sampled.drop_duplicates(
        subset=["diff_id"]
    )

    if len(sampled) < target_size:
        remaining = candidates[
            ~candidates["diff_id"].isin(
                sampled["diff_id"]
            )
        ]

        additional_size = min(
            target_size - len(sampled),
            len(remaining),
        )

        if additional_size > 0:
            additional = remaining.sample(
                n=additional_size,
                random_state=random_seed + 10_000,
            )

            sampled = pd.concat(
                [
                    sampled,
                    additional,
                ],
                ignore_index=False,
            )

    sampled = sampled.sample(
        frac=1,
        random_state=random_seed,
    )

    return sampled.head(
        target_size
    ).drop(
        columns=["annotation_stratum"],
        errors="ignore",
    )

In [32]:
annotation_parts = []

for split_name, target_size in LABEL_SETTINGS[
    "manual_review_targets"
].items():

    split_records = weak_label_manifest[
        weak_label_manifest[
            "dataset_split"
        ] == split_name
    ].copy()

    sampled_records = balanced_annotation_sample(
        dataframe=split_records,
        target_size=target_size,
        random_seed=(
            LABEL_SETTINGS["random_seed"]
            + {
                "train": 0,
                "validation": 100,
                "test": 200,
            }[split_name]
        ),
    )

    annotation_parts.append(
        sampled_records
    )

manual_annotation_queue = pd.concat(
    annotation_parts,
    ignore_index=True,
)

manual_annotation_queue.insert(
    0,
    "annotation_id",
    [
        f"DG-ANN-{index:05d}"
        for index in range(
            1,
            len(manual_annotation_queue) + 1
        )
    ],
)

manual_annotation_queue[
    "human_label"
] = ""

manual_annotation_queue[
    "human_rule_id"
] = ""

manual_annotation_queue[
    "review_status"
] = "pending"

manual_annotation_queue[
    "reviewer_confidence"
] = ""

manual_annotation_queue[
    "reviewer_notes"
] = ""

manual_annotation_queue[
    "include_in_gold_set"
] = ""

annotation_columns = [
    "annotation_id",
    "diff_id",
    "dataset_split",
    "repository",
    "commit_hash",
    "commit_author_date",
    "file_path",
    "configuration_type",
    "parser_mode",
    "operation",
    "field_path",
    "old_value",
    "new_value",
    "commit_message",
    "weak_label",
    "weak_label_confidence",
    "primary_rule_id",
    "matched_rule_ids",
    "rule_conflict",
    "weak_label_reason",
    "needs_manual_review",
    "human_label",
    "human_rule_id",
    "review_status",
    "reviewer_confidence",
    "reviewer_notes",
    "include_in_gold_set",
]

manual_annotation_queue = (
    manual_annotation_queue[
        annotation_columns
    ]
)

annotation_queue_path = (
    EVALUATION_DIR
    / "manual_annotation_queue.csv"
)

manual_annotation_queue.to_csv(
    annotation_queue_path,
    index=False,
)

for split_name in [
    "train",
    "validation",
    "test",
]:
    split_annotation_path = (
        EVALUATION_DIR
        / f"{split_name}_manual_annotation_queue.csv"
    )

    manual_annotation_queue[
        manual_annotation_queue[
            "dataset_split"
        ] == split_name
    ].to_csv(
        split_annotation_path,
        index=False,
    )

print(
    "Manual annotation queue size:",
    f"{len(manual_annotation_queue):,}",
)

display(
    manual_annotation_queue[
        "dataset_split"
    ]
    .value_counts()
    .rename_axis("dataset_split")
    .reset_index(name="records")
)

print("\nSaved to:")
print(annotation_queue_path)

Manual annotation queue size: 1,800


,dataset_split,records
0,train,700
1,test,700
2,validation,400



Saved to:
C:\Users\Lenovo\Desktop\DriftGuard\data\evaluation\manual_annotation_queue.csv


In [33]:
annotation_guidelines = """# DriftGuard Manual Annotation Guidelines

## Purpose

The weak label is only a suggestion. Review the actual field path, old value,
new value, operation, file type and commit message before assigning a human
label.

## Allowed human labels

- benign
- low
- medium
- high
- critical
- exclude

## Severity definitions

### benign

The change has no meaningful security or operational risk. Examples include
image-version updates, labels, names, descriptions and ordinary scaling
changes.

### low

The change involves a potentially sensitive field, but the security impact is
small, ambiguous or highly context-dependent.

### medium

The change weakens a protection or operational safeguard but does not directly
create unrestricted access. Examples include removed quotas, resource limits
or moderate policy weakening.

### high

The change is likely to create a serious security exposure. Examples include
insecure protocols, broad permissions, host namespace access or public
service exposure.

### critical

The change directly removes a core security control or introduces immediate
exploitation risk. Examples include disabled authentication, disabled TLS,
default credentials, privileged execution or unrestricted public CIDR access.

### exclude

Use this when the record is malformed, cannot be interpreted, is only a parser
artifact or lacks enough context to assign any severity.

## Gold-set rules

1. Do not automatically copy the weak label.
2. Review each row independently.
3. Set review_status to reviewed.
4. Set include_in_gold_set to yes only when the label is sufficiently clear.
5. Validation and test gold labels must never be used for model fitting.
6. Test labels must not be examined while tuning model thresholds.
"""

annotation_guidelines_path = (
    EVALUATION_DIR
    / "ANNOTATION_GUIDELINES.md"
)

annotation_guidelines_path.write_text(
    annotation_guidelines,
    encoding="utf-8",
)

print(
    "Saved annotation guidelines:",
    annotation_guidelines_path,
)

Saved annotation guidelines: C:\Users\Lenovo\Desktop\DriftGuard\data\evaluation\ANNOTATION_GUIDELINES.md


In [34]:
train_usage_values = set(
    weak_label_manifest.loc[
        weak_label_manifest[
            "dataset_split"
        ] == "train",
        "label_usage",
    ].unique()
)

validation_usage_values = set(
    weak_label_manifest.loc[
        weak_label_manifest[
            "dataset_split"
        ] == "validation",
        "label_usage",
    ].unique()
)

test_usage_values = set(
    weak_label_manifest.loc[
        weak_label_manifest[
            "dataset_split"
        ] == "test",
        "label_usage",
    ].unique()
)

split_usage_checks = {
    "Train records are weak-training candidates":
        train_usage_values
        == {"weak_training_candidate"},

    "Validation records are review-only":
        validation_usage_values
        == {"validation_review_only"},

    "Test records are review-only":
        test_usage_values
        == {"test_review_only"},
}

for check_name, passed in split_usage_checks.items():
    print(
        f"{'PASSED' if passed else 'FAILED':<8} | "
        f"{check_name}"
    )

assert all(split_usage_checks.values())

PASSED   | Train records are weak-training candidates
PASSED   | Validation records are review-only
PASSED   | Test records are review-only


In [35]:
expected_total_records = int(
    structural_outputs_df[
        "expected_records"
    ].sum()
)

valid_primary_rules = (
    weak_label_manifest[
        "primary_rule_id"
    ]
    .dropna()
    .isin(VALID_RULE_IDS)
    .all()
)

integrity_checks = {
    "Label manifest is not empty":
        len(weak_label_manifest) > 0,

    "Record count matches Notebook 03":
        len(weak_label_manifest)
        == expected_total_records,

    "Every Diff ID is present":
        weak_label_manifest[
            "diff_id"
        ].notna().all(),

    "Every Diff ID is unique":
        weak_label_manifest[
            "diff_id"
        ].is_unique,

    "Every weak label is valid":
        weak_label_manifest[
            "weak_label"
        ].isin(
            LABEL_SETTINGS[
                "valid_labels"
            ]
        ).all(),

    "Confidence values are between zero and one":
        weak_label_manifest[
            "weak_label_confidence"
        ].between(
            0,
            1,
            inclusive="both",
        ).all(),

    "All rule IDs are valid":
        valid_primary_rules,

    "Training labels exist":
        (
            weak_label_manifest[
                "dataset_split"
            ] == "train"
        ).any(),

    "Validation records exist":
        (
            weak_label_manifest[
                "dataset_split"
            ] == "validation"
        ).any(),

    "Unseen test records exist":
        (
            weak_label_manifest[
                "dataset_split"
            ] == "test"
        ).any(),

    "High-confidence training candidates exist":
        len(
            high_confidence_training_candidates
        ) > 0,

    "Manual annotation queue is not empty":
        len(manual_annotation_queue) > 0,
}

print("Security label integrity checks:\n")

for check_name, passed in integrity_checks.items():
    print(
        f"{'PASSED' if passed else 'FAILED':<8} | "
        f"{check_name}"
    )

all_checks_passed = (
    all(integrity_checks.values())
    and all(split_usage_checks.values())
)

print("\n" + "=" * 72)

if all_checks_passed:
    print("ALL SECURITY LABEL CHECKS PASSED")
else:
    print("ONE OR MORE SECURITY LABEL CHECKS FAILED")

print("=" * 72)

Security label integrity checks:

PASSED   | Label manifest is not empty
PASSED   | Record count matches Notebook 03
PASSED   | Every Diff ID is present
PASSED   | Every Diff ID is unique
PASSED   | Every weak label is valid
PASSED   | Confidence values are between zero and one
PASSED   | All rule IDs are valid
PASSED   | Training labels exist
PASSED   | Validation records exist
PASSED   | Unseen test records exist
PASSED   | High-confidence training candidates exist
PASSED   | Manual annotation queue is not empty

ALL SECURITY LABEL CHECKS PASSED


In [37]:
# Consolidated results required for reporting

overall_label_counts = (
    weak_label_manifest["weak_label"]
    .value_counts()
)

risky_labels = [
    "low",
    "medium",
    "high",
    "critical",
]

total_risky_records = int(
    overall_label_counts
    .reindex(risky_labels, fill_value=0)
    .sum()
)

total_benign_records = int(
    overall_label_counts.get("benign", 0)
)

total_unlabeled_records = int(
    overall_label_counts.get("unlabeled", 0)
)

high_confidence_training_count = len(
    high_confidence_training_candidates
)

manual_annotation_queue_size = len(
    manual_annotation_queue
)

print("=" * 72)
print("NOTEBOOK 04 — REQUIRED RESULTS")
print("=" * 72)

print("\n1. TOTAL LABEL COUNTS")
print("-" * 72)
print(
    "Risky records    :",
    f"{total_risky_records:,}",
)
print(
    "Benign records   :",
    f"{total_benign_records:,}",
)
print(
    "Unlabeled records:",
    f"{total_unlabeled_records:,}",
)

print("\nDetailed risk distribution:")

risk_distribution_report = pd.DataFrame(
    {
        "label": [
            "benign",
            "low",
            "medium",
            "high",
            "critical",
            "unlabeled",
        ],
        "records": [
            int(
                overall_label_counts.get(
                    label,
                    0,
                )
            )
            for label in [
                "benign",
                "low",
                "medium",
                "high",
                "critical",
                "unlabeled",
            ]
        ],
    }
)

risk_distribution_report["percentage"] = (
    risk_distribution_report["records"]
    / max(
        risk_distribution_report[
            "records"
        ].sum(),
        1,
    )
    * 100
)

display(risk_distribution_report)

print(
    "\n2. HIGH-CONFIDENCE TRAINING CANDIDATES"
)
print("-" * 72)
print(
    "Minimum confidence threshold:",
    LABEL_SETTINGS[
        "minimum_training_confidence"
    ],
)
print(
    "Candidate records            :",
    f"{high_confidence_training_count:,}",
)

high_confidence_distribution_report = (
    high_confidence_training_candidates[
        "weak_label"
    ]
    .value_counts()
    .reindex(
        [
            "benign",
            "low",
            "medium",
            "high",
            "critical",
        ],
        fill_value=0,
    )
    .rename_axis("weak_label")
    .reset_index(name="records")
)

display(
    high_confidence_distribution_report
)

print("\n3. RULE DISTRIBUTION")
print("-" * 72)

rule_distribution_report = (
    weak_label_manifest[
        weak_label_manifest[
            "primary_rule_id"
        ].notna()
    ]
    .groupby(
        [
            "primary_rule_id",
            "weak_label",
        ],
        as_index=False,
    )
    .agg(
        records=("diff_id", "count"),
        mean_confidence=(
            "weak_label_confidence",
            "mean",
        ),
        repositories=(
            "repository",
            "nunique",
        ),
    )
    .sort_values(
        [
            "records",
            "primary_rule_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

if rule_distribution_report.empty:
    print("No security-rule matches were generated.")
else:
    rule_distribution_report[
        "mean_confidence"
    ] = (
        rule_distribution_report[
            "mean_confidence"
        ]
        .round(4)
    )

    display(rule_distribution_report)

print("\n4. MANUAL ANNOTATION QUEUE")
print("-" * 72)
print(
    "Total records selected:",
    f"{manual_annotation_queue_size:,}",
)

manual_queue_distribution_report = (
    manual_annotation_queue
    .groupby(
        [
            "dataset_split",
            "weak_label",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "records",
        }
    )
    .sort_values(
        [
            "dataset_split",
            "records",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(
    manual_queue_distribution_report
)

print("\n5. FAILED INTEGRITY CHECKS")
print("-" * 72)

combined_integrity_checks = {}

if "integrity_checks" in globals():
    combined_integrity_checks.update(
        {
            f"Data integrity: {name}": passed
            for name, passed
            in integrity_checks.items()
        }
    )

if "split_usage_checks" in globals():
    combined_integrity_checks.update(
        {
            f"Split usage: {name}": passed
            for name, passed
            in split_usage_checks.items()
        }
    )

failed_integrity_checks = [
    check_name
    for check_name, passed
    in combined_integrity_checks.items()
    if not bool(passed)
]

if failed_integrity_checks:
    print(
        "FAILED CHECKS:",
        len(failed_integrity_checks),
    )

    for check_number, check_name in enumerate(
        failed_integrity_checks,
        start=1,
    ):
        print(
            f"{check_number}. {check_name}"
        )
else:
    print("No failed integrity checks.")
    print("All integrity and split-usage checks passed.")

print("\n" + "=" * 72)
print("REQUIRED RESULTS GENERATED")
print("=" * 72)

NOTEBOOK 04 — REQUIRED RESULTS

1. TOTAL LABEL COUNTS
------------------------------------------------------------------------
Risky records    : 109,138
Benign records   : 221,155
Unlabeled records: 29,445

Detailed risk distribution:


,label,records,percentage
0,benign,221155,61.476686
1,low,105782,29.405289
2,medium,430,0.119531
3,high,1380,0.383613
4,critical,1546,0.429757
5,unlabeled,29445,8.185124



2. HIGH-CONFIDENCE TRAINING CANDIDATES
------------------------------------------------------------------------
Minimum confidence threshold: 0.85
Candidate records            : 44,733


,weak_label,records
0,benign,42657
1,low,0
2,medium,273
3,high,976
4,critical,827



3. RULE DISTRIBUTION
------------------------------------------------------------------------


,primary_rule_id,weak_label,records,mean_confidence,repositories
9,UNCLASSIFIED_SENSITIVE_FIELD_CHANGE,low,105782,0.7600,6
0,AUTH_REMOVED,critical,1175,0.9800,5
4,OPEN_PORT_EXPOSURE,high,643,0.9600,6
7,RESOURCE_LIMIT_REMOVED,medium,430,0.9100,4
6,PERMISSIVE_ACL,high,382,0.9400,4
2,INSECURE_PROTOCOL,high,355,0.9500,5
1,DEFAULT_CREDENTIALS,critical,165,0.9900,5
8,TLS_DISABLED,critical,111,0.9822,4
3,OPEN_PORT_EXPOSURE,critical,76,0.9900,3
5,PERMISSIVE_ACL,critical,19,0.9800,3



4. MANUAL ANNOTATION QUEUE
------------------------------------------------------------------------
Total records selected: 1,800


,dataset_split,weak_label,records
3,test,low,202
0,test,benign,168
5,test,unlabeled,122
2,test,high,92
1,test,critical,70
4,test,medium,46
9,train,low,192
6,train,benign,151
11,train,unlabeled,124
8,train,high,96



5. FAILED INTEGRITY CHECKS
------------------------------------------------------------------------
No failed integrity checks.
All integrity and split-usage checks passed.

REQUIRED RESULTS GENERATED


In [38]:
overall_label_counts = (
    weak_label_manifest[
        "weak_label"
    ]
    .value_counts()
)

risky_records = int(
    overall_label_counts.get("low", 0)
    + overall_label_counts.get("medium", 0)
    + overall_label_counts.get("high", 0)
    + overall_label_counts.get("critical", 0)
)

high_confidence_train_count = len(
    high_confidence_training_candidates
)

manual_review_count = int(
    weak_label_manifest[
        "needs_manual_review"
    ].sum()
)

print("=" * 72)
print("NOTEBOOK 04 COMPLETED")
print("=" * 72)

print(
    "Total field-level records       :",
    f"{len(weak_label_manifest):,}",
)

print(
    "Weakly labeled risky records    :",
    f"{risky_records:,}",
)

print(
    "Weakly labeled benign records   :",
    f"{overall_label_counts.get('benign', 0):,}",
)

print(
    "Unlabeled records               :",
    f"{overall_label_counts.get('unlabeled', 0):,}",
)

print(
    "High-confidence train candidates:",
    f"{high_confidence_train_count:,}",
)

print(
    "Records marked for review       :",
    f"{manual_review_count:,}",
)

print(
    "Manual annotation queue         :",
    f"{len(manual_annotation_queue):,}",
)

print("\nLabel distribution:")
display(
    overall_label_counts
    .rename_axis("weak_label")
    .reset_index(name="records")
)

print("\nImportant:")
print("- Weak labels are not treated as final ground truth.")
print("- Only training-repository labels may be used for fitting.")
print("- Validation and test labels remain review-only.")
print("- Final metrics require manually reviewed gold labels.")

print("\nNext notebook:")
print("05_dataset_cleaning_and_validation.ipynb")

NOTEBOOK 04 COMPLETED
Total field-level records       : 359,738
Weakly labeled risky records    : 109,138
Weakly labeled benign records   : 221,155
Unlabeled records               : 29,445
High-confidence train candidates: 44,733
Records marked for review       : 294,630
Manual annotation queue         : 1,800

Label distribution:


,weak_label,records
0,benign,221155
1,low,105782
2,unlabeled,29445
3,critical,1546
4,high,1380
5,medium,430



Important:
- Weak labels are not treated as final ground truth.
- Only training-repository labels may be used for fitting.
- Validation and test labels remain review-only.
- Final metrics require manually reviewed gold labels.

Next notebook:
05_dataset_cleaning_and_validation.ipynb
